## Annotate the new data's latent scores with pathways and drugs

Links each new sample's BioBombe latent scores to the same GSEA pathway
results and drug correlations used everywhere else in this project, via
`utils/pinwheels.py`'s `compute_and_plot_latent_scores` /
`assign_unique_latent_dims` — the same functions
`6.collab-data/scripts/3.pinwheel_plots.py` uses for collaborator data.

Run this script with `8.apply-biobombe-new-data/` as the working directory.

In [1]:
import pathlib
import sys

import pandas as pd

sys.path.insert(0, "../utils/")
from pinwheels import assign_unique_latent_dims, compute_and_plot_latent_scores

In [2]:
data_results_dir = pathlib.Path("data/largaespada/results")

combined_latent_df = pd.read_parquet(data_results_dir / "NF1_data_latent_scores.parquet")
print(f"Latent scores: {combined_latent_df.shape}")

Latent scores: (3888, 408)


### Pathway annotation: CORUM and Reactome, kept separate

A real, ensemble-wide Reactome run did happen for this project at some
point — 4.drug-dependency/results/all_reactome_results.parquet holds genuine
Reactome pathways (R-HSA-tagged names, e.g. "rRNA Modification In Nucleus
And Cytosol R-HSA-6790901") spanning all 6 model types, so this isn't
hypothetical. What's missing is the raw, per-latent-dimension GSEA hit table
that would have fed it (the direct analog of combined_z_matrix_gsea_results.parquet,
but for Reactome) — it isn't in 3.run-biobombe/gsea_results/ or anywhere
else in this repo's git history. `reactome_gsea_path` below is where
`2.interpret_biobombe_with_gsea.py` would need to write one (by re-running
it with a Reactome library) for that half to populate again.

`3.run-biobombe/gsea_results/combined_z_matrix_gsea_results.parquet` — the
one raw file that does exist — is CORUM, confirmed directly: every one of
its 170 pathway names matches CORUM's complex-naming convention ("X complex
(human)"), never Reactome's R-HSA- IDs. `4.drug-dependency/scripts/7.pinwheel_plots.py`
actually has this backwards (it labels this file `reactome_df` and expects
a nonexistent "_corum"-suffixed file for CORUM) — that script has never
successfully run against its current code, since the file it wants for
CORUM was never committed either.

Both results, when present, use a column literally called "reactome_pathway"
for the pathway name — that's `2.interpret_biobombe_with_gsea.py`'s schema
regardless of which library was actually run, not a labeling mistake here.

### Pathway annotation (CORUM GSEA results)

In [3]:
gsea_sources = {
    "CORUM": pathlib.Path("../3.run-biobombe/gsea_results/combined_z_matrix_gsea_results.parquet"),
    "Reactome": pathlib.Path("../3.run-biobombe/gsea_results/combined_z_matrix_gsea_results_reactome.parquet"),
}

pathway_max_by_library = {}
for library, gsea_path in gsea_sources.items():
    if not gsea_path.exists():
        print(f"{library}: {gsea_path} not found, skipping.")
        continue

    gsea_df = pd.read_parquet(gsea_path)
    # Drop the shuffled negative-control rows and keep only nominally significant
    # hits. This file has no fdr column (unlike the single-VAE GSEA results
    # elsewhere in the project), so significance here is by p_value alone.
    significant_gsea_df = gsea_df[(~gsea_df["shuffled"]) & (gsea_df["p_value"] < 0.05)].copy()
    print(f"{library}: {significant_gsea_df.shape[0]} / {gsea_df.shape[0]} significant hits")

    # nes_score (normalized enrichment score), not gsea_es_score: NES adjusts
    # for gene-set-size effects, and it's what the previously-run ensemble-wide
    # annotation (4.drug-dependency/results/all_corum_results.parquet /
    # all_reactome_results.parquet) actually used.
    pathway_max_by_library[library] = assign_unique_latent_dims(
        significant_gsea_df, score_col="nes_score", target_col="reactome_pathway"
    )

CORUM: 52386 / 1142910 significant hits
Processing pathway: C complex spliceosome (human)
Assigned latent dimension 2 for model vanillavae, 18, init 2 to pathway C complex spliceosome (human).
Processing pathway: Mediator complex (human)
Assigned latent dimension 2 for model betavae, 10, init 1 to pathway Mediator complex (human).
Processing pathway: Spliceosome (human)
Assigned latent dimension 2 for model betavae, 3, init 0 to pathway Spliceosome (human).
Processing pathway: DAB complex (human)
Assigned latent dimension 5 for model betatcvae, 12, init 1 to pathway DAB complex (human).
Processing pathway: DA complex (human)


Assigned latent dimension 45 for model nmf, 200, init 0 to pathway DA complex (human).
Processing pathway: BRM-SIN3A-HDAC complex (human)
Assigned latent dimension 2 for model betatcvae, 2, init 0 to pathway BRM-SIN3A-HDAC complex (human).
Processing pathway: STAGA complex (SPT3-TAF9-GCN5 acetyltransferase complex) (human)
Assigned latent dimension 40 for model betatcvae, 50, init 1 to pathway STAGA complex (SPT3-TAF9-GCN5 acetyltransferase complex) (human).
Processing pathway: Polycystin-1 multiprotein complex (ACTN1, CDH1, SRC, JUP, VCL, CTNNB1, PXN, BCAR1, PKD1, PTK2, TLN1) (human)
Assigned latent dimension 35 for model betatcvae, 80, init 4 to pathway Polycystin-1 multiprotein complex (ACTN1, CDH1, SRC, JUP, VCL, CTNNB1, PXN, BCAR1, PKD1, PTK2, TLN1) (human).
Processing pathway: mRNA decay complex (UPF1, UPF2, UPF3B, DCP2, XRN1, XRN2, EXOSC2, EXOSC4, EXOSC10, PARN) (human)
Assigned latent dimension 5 for model betatcvae, 16, init 4 to pathway mRNA decay complex (UPF1, UPF2, UPF3B, 

Assigned latent dimension 16 for model vanillavae, 200, init 0 to pathway BRAFT complex (human).
Processing pathway: NuA4/Tip60-HAT complex B (human)
Assigned latent dimension 3 for model betatcvae, 35, init 1 to pathway NuA4/Tip60-HAT complex B (human).
Processing pathway: Nup 107-160 subcomplex (human)
Assigned latent dimension 4 for model betatcvae, 9, init 4 to pathway Nup 107-160 subcomplex (human).
Processing pathway: DRIP complex (human)
Assigned latent dimension 16 for model betatcvae, 200, init 4 to pathway DRIP complex (human).
Processing pathway: BAF complex (human)
Assigned latent dimension 100 for model betatcvae, 150, init 3 to pathway BAF complex (human).
Processing pathway: ARC-L complex (human)
Assigned latent dimension 3 for model betatcvae, 10, init 2 to pathway ARC-L complex (human).
Processing pathway: cMYC-ATPase-helicase complex (human)
Assigned latent dimension 9 for model betatcvae, 18, init 3 to pathway cMYC-ATPase-helicase complex (human).
Processing pathway:

Assigned latent dimension 35 for model nmf, 90, init 0 to pathway EBAFa complex (human).
Processing pathway: CRM1-RAN-PHAX-CBC complex (cap binding complex) (human)
Assigned latent dimension 18 for model betatcvae, 80, init 4 to pathway CRM1-RAN-PHAX-CBC complex (cap binding complex) (human).
Processing pathway: SIN3-SAP25 complex (human)
Assigned latent dimension 150 for model betatcvae, 150, init 0 to pathway SIN3-SAP25 complex (human).
Processing pathway: LARC complex (LCR-associated remodeling complex) (human)
Assigned latent dimension 12 for model betavae, 30, init 4 to pathway LARC complex (LCR-associated remodeling complex) (human).
Processing pathway: Conserved oligomeric Golgi (COG) complex (human)
Assigned latent dimension 14 for model betatcvae, 60, init 0 to pathway Conserved oligomeric Golgi (COG) complex (human).
Processing pathway: BRD4 complex (human)
Assigned latent dimension 70 for model betatcvae, 90, init 2 to pathway BRD4 complex (human).
Processing pathway: Respir

In [4]:
pathway_results = []
for library, pathway_max in pathway_max_by_library.items():
    for model_id in combined_latent_df["ModelID"].unique():
        df = compute_and_plot_latent_scores(
            model_id, combined_latent_df, pathway_max, "reactome_pathway", "nes_score",
            library, make_plot=False,
        )
        df["gene_set_library"] = library
        pathway_results.append(df)

pathway_results_df = pd.concat(pathway_results, ignore_index=True) if pathway_results else pd.DataFrame()
print(f"Pathway annotations (CORUM + Reactome combined): {pathway_results_df.shape}")

S462
ST88-14


STS-26T
iHSC1L_C3
iHSC1L_F12


iPN1L
iPN2J
iPN3D
Pathway annotations (CORUM + Reactome combined): (1235, 11)


### Drug annotation

This step needs `4.drug-dependency/results/combined_latent_drug_correlations.parquet`,
the ensemble-wide drug correlation table `4.drug-dependency/scripts/5.drug_latent_dim_correlation.py`
produces. That script depends on `saved_models/`, which only exists locally
now — if it hasn't been run yet, drug annotation is skipped here rather than failing.

In [5]:
drug_correlation_path = pathlib.Path("../4.drug-dependency/results/combined_latent_drug_correlations.parquet")

if drug_correlation_path.exists():
    drug_df = pd.read_parquet(drug_correlation_path)
    significant_drug_df = drug_df[
        (~drug_df["shuffled"]) & (drug_df["pearson_correlation"].abs() > 0.15) & (drug_df["p_value"] < 0.05)
    ].copy()
    drug_max = assign_unique_latent_dims(
        significant_drug_df, score_col="pearson_correlation", target_col="name"
    )

    drug_results = []
    for model_id in combined_latent_df["ModelID"].unique():
        df = compute_and_plot_latent_scores(
            model_id, combined_latent_df, drug_max, "name", "pearson_correlation", "Drug",
            make_plot=False,
        )
        drug_results.append(df)
    drug_results_df = pd.concat(drug_results, ignore_index=True) if drug_results else pd.DataFrame()
    print(f"Drug annotations: {drug_results_df.shape}")
else:
    drug_results_df = pd.DataFrame()
    print(
        f"{drug_correlation_path} not found — skipping drug annotation. "
        "Run 4.drug-dependency/scripts/5.drug_latent_dim_correlation.py first to enable it."
    )

Processing pathway: idasanutlin
Assigned latent dimension 4 for model nmf, 25, init 0 to pathway idasanutlin.
Processing pathway: AMG-232
Assigned latent dimension 8 for model ica, 10, init 0 to pathway AMG-232.
Processing pathway: CGM097
Assigned latent dimension 29 for model nmf, 45, init 0 to pathway CGM097.
Processing pathway: XL-647
Assigned latent dimension 9 for model nmf, 50, init 0 to pathway XL-647.
Processing pathway: nutlin-3
Assigned latent dimension 29 for model nmf, 30, init 0 to pathway nutlin-3.
Processing pathway: poziotinib
Assigned latent dimension 10 for model nmf, 18, init 0 to pathway poziotinib.
Processing pathway: atiprimod
Assigned latent dimension 72 for model ica, 100, init 0 to pathway atiprimod.
Processing pathway: ibrutinib
Assigned latent dimension 6 for model nmf, 12, init 0 to pathway ibrutinib.
Processing pathway: AZ-628
Assigned latent dimension 16 for model nmf, 60, init 0 to pathway AZ-628.
Processing pathway: LGX818
Assigned latent dimension 19 fo

Assigned latent dimension 4 for model pca, 5, init 0 to pathway BMS-690514.
Processing pathway: SB-590885
Assigned latent dimension 16 for model nmf, 50, init 0 to pathway SB-590885.
Processing pathway: PD-318088
Assigned latent dimension 13 for model ica, 60, init 0 to pathway PD-318088.
Processing pathway: AS-703026
Assigned latent dimension 1 for model ica, 20, init 0 to pathway AS-703026.
Processing pathway: selumetinib
Assigned latent dimension 22 for model ica, 50, init 0 to pathway selumetinib.
Processing pathway: BVD-523
Assigned latent dimension 3 for model nmf, 4, init 0 to pathway BVD-523.
Processing pathway: refametinib
Assigned latent dimension 32 for model nmf, 40, init 0 to pathway refametinib.
Processing pathway: NVP-BHG712
Assigned latent dimension 16 for model ica, 35, init 0 to pathway NVP-BHG712.
Processing pathway: U-0126
Assigned latent dimension 13 for model nmf, 45, init 0 to pathway U-0126.
Processing pathway: EVP4593
Assigned latent dimension 0 for model nmf, 

Assigned latent dimension 3 for model nmf, 12, init 0 to pathway PF-562271.
Processing pathway: PHA-680632
Assigned latent dimension 30 for model nmf, 60, init 0 to pathway PHA-680632.
Processing pathway: SB-205384
Assigned latent dimension 3 for model ica, 40, init 0 to pathway SB-205384.
Processing pathway: AZ3146
Assigned latent dimension 3 for model nmf, 14, init 0 to pathway AZ3146.
Processing pathway: teroxirone
Assigned latent dimension 90 for model nmf, 100, init 0 to pathway teroxirone.
Processing pathway: alpelisib
Assigned latent dimension 21 for model nmf, 40, init 0 to pathway alpelisib.
Processing pathway: beta-lapachone
Assigned latent dimension 13 for model nmf, 30, init 0 to pathway beta-lapachone.
Processing pathway: mechlorethamine
Assigned latent dimension 69 for model ica, 80, init 0 to pathway mechlorethamine.
Processing pathway: NU6027
Assigned latent dimension 24 for model ica, 45, init 0 to pathway NU6027.
Processing pathway: WZ-3146
Assigned latent dimension 1

Assigned latent dimension 43 for model nmf, 70, init 0 to pathway CGS-15943.
Processing pathway: drospirenone
Assigned latent dimension 49 for model nmf, 50, init 0 to pathway drospirenone.
Processing pathway: FLI-06
Assigned latent dimension 48 for model nmf, 70, init 0 to pathway FLI-06.
Processing pathway: doxycycline
Assigned latent dimension 15 for model nmf, 25, init 0 to pathway doxycycline.
Processing pathway: pyronaridine
Assigned latent dimension 5 for model pca, 9, init 0 to pathway pyronaridine.
Processing pathway: serdemetan
Assigned latent dimension 8 for model pca, 10, init 0 to pathway serdemetan.
Processing pathway: idoxuridine
Assigned latent dimension 11 for model ica, 18, init 0 to pathway idoxuridine.
Processing pathway: bardoxolone-methyl
Assigned latent dimension 97 for model ica, 150, init 0 to pathway bardoxolone-methyl.
Processing pathway: xanomeline
Assigned latent dimension 53 for model ica, 60, init 0 to pathway xanomeline.
Processing pathway: SCH-900776
As

Assigned latent dimension 25 for model ica, 30, init 0 to pathway doxercalciferol.
Processing pathway: amidopyrine
Assigned latent dimension 29 for model nmf, 40, init 0 to pathway amidopyrine.
Processing pathway: PF-05212384
Assigned latent dimension 23 for model nmf, 30, init 0 to pathway PF-05212384.
Processing pathway: paraxanthine
Assigned latent dimension 62 for model ica, 90, init 0 to pathway paraxanthine.
Processing pathway: tyrphostin-A9
Assigned latent dimension 28 for model pca, 60, init 0 to pathway tyrphostin-A9.
Processing pathway: delanzomib
Assigned latent dimension 3 for model pca, 18, init 0 to pathway delanzomib.
Processing pathway: mepacrine
Assigned latent dimension 0 for model nmf, 16, init 0 to pathway mepacrine.
Processing pathway: AMG458
Assigned latent dimension 84 for model nmf, 100, init 0 to pathway AMG458.
Processing pathway: GSK429286A
Assigned latent dimension 1 for model pca, 14, init 0 to pathway GSK429286A.
Processing pathway: RGFP966
Assigned latent

Assigned latent dimension 173 for model ica, 200, init 0 to pathway thioguanine.
Processing pathway: PHA-665752
Assigned latent dimension 90 for model nmf, 150, init 0 to pathway PHA-665752.
Processing pathway: sunitinib
Assigned latent dimension 96 for model nmf, 200, init 0 to pathway sunitinib.
Processing pathway: GSK-J4
Assigned latent dimension 54 for model ica, 200, init 0 to pathway GSK-J4.
Processing pathway: cyamemazine
Assigned latent dimension 16 for model nmf, 80, init 0 to pathway cyamemazine.
Processing pathway: valnemulin
Assigned latent dimension 2 for model nmf, 20, init 0 to pathway valnemulin.
Processing pathway: tioxolone
Assigned latent dimension 7 for model nmf, 10, init 0 to pathway tioxolone.
Processing pathway: ICG-001
Assigned latent dimension 23 for model nmf, 150, init 0 to pathway ICG-001.
Processing pathway: crenolanib
Assigned latent dimension 68 for model pca, 90, init 0 to pathway crenolanib.
Processing pathway: WYE-125132
Assigned latent dimension 44 f

... (14 outputs left out)

No unused latent dimensions left for pathway fluspirilene.
Processing pathway: C-1
No unused latent dimensions left for pathway C-1.
Processing pathway: oxiconazole
Assigned latent dimension 34 for model ica, 100, init 0 to pathway oxiconazole.
Processing pathway: gliclazide
No unused latent dimensions left for pathway gliclazide.
Processing pathway: acipimox
No unused latent dimensions left for pathway acipimox.
Processing pathway: 1-deoxymannojirimycin
No unused latent dimensions left for pathway 1-deoxymannojirimycin.
Processing pathway: beta-CCB
No unused latent dimensions left for pathway beta-CCB.
Processing pathway: yohimbine
No unused latent dimensions left for pathway yohimbine.
Processing pathway: buspirone
No unused latent dimensions left for pathway buspirone.
Processing pathway: urapidil
No unused latent dimensions left for pathway urapidil.
Processing pathway: JX-401
No unused latent dimensions left for pathway JX-401.
Processing pathway: LX1031
Assigned latent dimension 

S462
ST88-14


STS-26T
iHSC1L_C3
iHSC1L_F12


iPN1L
iPN2J
iPN3D
Drug annotations: (21864, 10)


### Save top associations per sample

In [6]:
top_pathways_per_sample = (
    pathway_results_df.sort_values(["ModelID", "gene_set_library", "pathway_score"], ascending=[True, True, False])
    .groupby(["ModelID", "gene_set_library"])
    .head(20)
    if not pathway_results_df.empty else pathway_results_df
)
top_pathways_per_sample.to_parquet(data_results_dir / "NF1_data_top_pathways.parquet", index=False)

if not drug_results_df.empty:
    top_drugs_per_sample = (
        drug_results_df.sort_values(["ModelID", "pathway_score"], ascending=[True, False])
        .groupby("ModelID")
        .head(20)
    )
    top_drugs_per_sample.to_parquet(data_results_dir / "NF1_data_top_drugs.parquet", index=False)

print(f"Saved annotated results to {data_results_dir}")

Saved annotated results to data/largaespada/results
